In [2]:
library(tidyverse)
library(dbplyr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




In [24]:
# Read in data
data <- read.csv('data/additional_destinations_wide_detail.csv', header = TRUE)

In [25]:
head(data)

,person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,NA,NA,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
2,00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2015/2016,Education,Education,Education,NA,NA,Employment,Employment,NA,Employment,Employment,Employment,Employment
3,000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,2015/2016,NA,Education,Education,Education,NA,Education,Education,NA,Education,Education,NA,NA
4,00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,2016/2017,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA
5,0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017,Education,Education,Education,Education,Education,Education,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed
6,00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2016/2017,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA


## Prep data

In [26]:
data |>
    nrow()

[1] 17742

In [27]:
data |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17742


In [28]:
data |>
    group_by(person_id) |>
    filter(n() > 1)

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [29]:
# drop all NAs 
destinations_seq <- data |>
    filter(!if_all(`X9`:`X8`, is.na))

In [30]:
destinations_seq |>
    nrow()

[1] 17538

## add activity counts per row

In [31]:
destinations_counts <- destinations_seq |>
    rowwise() |>
    mutate(
    Education = sum(c_across(`X9`:`X8`) == "Education", na.rm = TRUE),
    Employment = sum(c_across(`X9`:`X8`) == "Employment", na.rm = TRUE),
    Training = sum(c_across(`X9`:`X8`) == "Training", na.rm = TRUE),
    Refused = sum(c_across(`X9`:`X8`) == "Refused", na.rm = TRUE),
    # check starts with 
    NEET = sum(grepl("^NEET", c_across(`X9`:`X8`)), na.rm = TRUE),
    NA_count = sum(is.na(c_across(`X9`:`X8`)))
        ) |>
    ungroup()

## Classify records

In [32]:
destinations_counts |>
 filter(str_starts(X9, 'NEET') & (Education == 11 | Employment == 11 | Training == 11))

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>
0772E99CADC7123F2940B90F89283CA8AFD36E0D99F083D845B5612729EECA71,2016/2017,NEET: Not ready,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,0,11,0,0,1,0
1314C053A3BA094882EC91EF74989449A1EAFCBAEA75054B7C490BB2521CF468,2016/2017,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,Training,Training,Training,Training,Training,0,0,11,0,1,0
161CDCAC6F2A4F846BB87AF3CA0496574BDBCDA96A1DAA8F17D4DCE0FA6DB3EB,2015/2016,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,Training,Training,Training,Training,Training,0,0,11,0,1,0
17623045CF888706B6604C1B734B26948DED4794081F8369559D926DA3AA3C77,2015/2016,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,11,0,0,0,1,0
19CB3F364B8534C419E6C91A2DF2000E806A3E2E691700914297EC9DE3CEE53B,2016/2017,NEET: Seeking EET,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,Employment,0,11,0,0,1,0
20121FDDC7BC077F1757E368C10056CCD4EEAAD117E0EFAF8183F90BCFC1D2C8,2016/2017,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,11,0,0,0,1,0
22FCCC4C1C2D332BF32E47283FDDA31D3693A659330583B6599B0036ED7B8AD0,2015/2016,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,11,0,0,0,1,0
2B3B49BD5A300A48C664A70EB538880F608F8D429AD8D7CEFB78A49B849F4554,2015/2016,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,11,0,0,0,1,0
57B2F831424726F457EBB4AD49B2A8BF3DB54BC4BF3CC4565CF8FA2DE4B9BEAB,2016/2017,NEET: Teen parent,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,11,0,0,0,1,0


In [33]:
destinations_counts |> 
    group_by(person_id) |>
    filter(n() > 1)

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>


In [34]:
destinations_counts |>
    filter(NEET == 1)

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>
007570ED5702694AEE71FBCA3ED93584AD7C19E06FE4CB64203ED8BE71B024EA,2015/2016,Education,NA,Education,Education,Education,Education,NEET: Seeking EET,NA,Employment,Employment,NA,Employment,5,3,0,0,1,3
013B9BA4BD97CDC7AE586437E9E9BB2C0F9C0A199AB5AB5B52F8B12D5E93A9A0,2015/2016,Education,NA,NA,NEET: Seeking EET,Training,Training,NA,Training,Training,NA,Training,Training,1,0,6,0,1,4
0145A692871133E73206BF7AEEA5D85D1E03E9876C5FC24BEEFFBCEC29566F1C,2015/2016,NA,NA,NA,NA,NEET: Seeking EET,Employment,Employment,Employment,NA,Employment,Employment,Employment,0,6,0,0,1,5
01C1CC6E8870E9159870D4960A30C011C7856B98A4E14EDF5697848D0C7857BC,2016/2017,Education,Education,NA,NA,NA,NA,NA,NA,NEET: Start date agreed,Employment,Employment,Employment,2,3,0,0,1,6
0295926630EA8CABA68406F62356E89F054D4A130CED06185F9B541E3C89B7E7,2016/2017,NA,Education,Education,Education,Education,Education,Education,Education,Education,NEET: Seeking EET,Employment,Employment,8,2,0,0,1,1
0383BAC74CDF01BD05413B510B2961DF66D212EAF8EFBD4705FA540B501451F1,2016/2017,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,Training,Training,Training,Training,NA,0,0,10,0,1,1
03874AA24F3E1A774036B356A7D82C9842E2E7B4BE068CE06C2424910ACD6EE6,2015/2016,Education,Education,Education,Education,Education,Education,Education,NEET: Seeking EET,Training,Training,Training,Training,7,0,4,0,1,0
03C67308298F1CB39B63B21FE0EA24C3AB3DD9B4062968764C5A183CB9AF3C82,2016/2017,Education,Education,Education,Education,Education,NEET: Seeking EET,Education,Education,Education,NA,Education,Education,10,0,0,0,1,1
03FB0EC85857ECEAA9D03E0855E5981B9BE3E551FC939A8957962EBC34DC49BE,2015/2016,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,NEET: Seeking EET,11,0,0,0,1,0


In [35]:
destinations_counts_labelled <- destinations_counts |>
    mutate(Label = case_when(
        NEET == 0 & Refused == 0 & NA_count <= 4 ~ 'Steady EET',
        str_starts(X9, 'NEET') & (Education == 11 | Employment == 11 | Training == 11) ~ 'Steady EET',
        NEET == 0 & Refused == 0 & NA_count >= 5 ~ 'DROP',
        TRUE ~ 'Risky trajectory'
    )
          )

In [36]:
head(destinations_counts_labelled)

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,10,0,0,0,0,2,Steady EET
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2015/2016,Education,Education,Education,NA,NA,Employment,Employment,NA,⋯,Employment,Employment,Employment,3,6,0,0,0,3,Steady EET
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,2015/2016,NA,Education,Education,Education,NA,Education,Education,NA,⋯,Education,NA,NA,7,0,0,0,0,5,DROP
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,2016/2017,Education,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,NA,11,0,0,0,0,1,Steady EET
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017,Education,Education,Education,Education,Education,Education,NEET: Start date agreed,NEET: Start date agreed,⋯,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed,6,0,0,0,6,0,Risky trajectory
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2016/2017,Education,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,NA,11,0,0,0,0,1,Steady EET


In [37]:
destinations_counts_labelled |>
    filter(Label == 'Risky trajectory')

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017,Education,Education,Education,Education,Education,Education,NEET: Start date agreed,NEET: Start date agreed,⋯,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed,6,0,0,0,6,0,Risky trajectory
00247728E0E765C3F36CF7DBDF2A3509DB16E5CED7E4707969748F4B62DA78A9,2015/2016,Education,Education,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,2,0,0,0,10,0,Risky trajectory
0045ADA48CA299BC4445318E8D951610DCB89A829665CFC821E92340F0F8B30D,2015/2016,NEET: Seeking EET,NEET: Seeking EET,Training,Training,Training,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,0,0,3,0,9,0,Risky trajectory
007570ED5702694AEE71FBCA3ED93584AD7C19E06FE4CB64203ED8BE71B024EA,2015/2016,Education,NA,Education,Education,Education,Education,NEET: Seeking EET,NA,⋯,Employment,NA,Employment,5,3,0,0,1,3,Risky trajectory
013B9BA4BD97CDC7AE586437E9E9BB2C0F9C0A199AB5AB5B52F8B12D5E93A9A0,2015/2016,Education,NA,NA,NEET: Seeking EET,Training,Training,NA,Training,⋯,NA,Training,Training,1,0,6,0,1,4,Risky trajectory
0145A692871133E73206BF7AEEA5D85D1E03E9876C5FC24BEEFFBCEC29566F1C,2015/2016,NA,NA,NA,NA,NEET: Seeking EET,Employment,Employment,Employment,⋯,Employment,Employment,Employment,0,6,0,0,1,5,Risky trajectory
017BD83D33E24C2CE047BFB67D7D010694F10CAF3E02B4360C88D4EE29E119FD,2015/2016,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,⋯,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,0,0,0,0,12,0,Risky trajectory
01C1CC6E8870E9159870D4960A30C011C7856B98A4E14EDF5697848D0C7857BC,2016/2017,Education,Education,NA,NA,NA,NA,NA,NA,⋯,Employment,Employment,Employment,2,3,0,0,1,6,Risky trajectory
01D4C5437E5A02C67E1473FEF63139486A1A85EA656181F6B68633068E174AEB,2015/2016,Education,Education,Education,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,Training,⋯,Training,Training,Training,3,0,5,0,4,0,Risky trajectory


In [38]:
destinations_counts_labelled |>
    filter(str_starts(X9, 'NEET'))

person_id,NCCIS_ACADYR,X9,X10,X11,X12,X1,X2,X3,X4,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
0045ADA48CA299BC4445318E8D951610DCB89A829665CFC821E92340F0F8B30D,2015/2016,NEET: Seeking EET,NEET: Seeking EET,Training,Training,Training,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,0,0,3,0,9,0,Risky trajectory
017BD83D33E24C2CE047BFB67D7D010694F10CAF3E02B4360C88D4EE29E119FD,2015/2016,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,⋯,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,0,0,0,0,12,0,Risky trajectory
02340F3F5D13AB36293E7F6DF375CE9AAE460153C9B2F3682ABF23C881A11830,2016/2017,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NA,NA,NA,NA,⋯,NA,NA,NA,0,0,0,0,4,8,Risky trajectory
0383BAC74CDF01BD05413B510B2961DF66D212EAF8EFBD4705FA540B501451F1,2016/2017,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,Training,⋯,Training,Training,NA,0,0,10,0,1,1,Risky trajectory
056129E5E118171D81B2BC0BD4890FB5852504EE358F6FB85D40F7104DB6F32E,2015/2016,NEET: Seeking EET,Employment,Employment,Employment,Employment,Employment,Employment,Training,⋯,Training,Training,Training,0,6,5,0,1,0,Risky trajectory
0614C169813351BB7F8E6599CC1AA658566641F7868CE4309AB02F0FC73141E2,2015/2016,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,Training,⋯,Training,NA,NA,0,0,9,0,1,2,Risky trajectory
061E7BFF81DD6F6752412B82E1852C350767D72B1CD7A39704F0C89611560F0F,2016/2017,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,Training,⋯,Employment,Employment,Employment,0,4,7,0,1,0,Risky trajectory
069DD966BBEF82F3A31A3DCA9F94672B808AD8938C58CAA6D4E9D14F5FDE23E6,2015/2016,NEET: Start date agreed,Training,Training,Training,NA,Education,Education,Education,⋯,Education,Education,Education,7,0,3,0,1,1,Risky trajectory
0772E99CADC7123F2940B90F89283CA8AFD36E0D99F083D845B5612729EECA71,2016/2017,NEET: Not ready,Employment,Employment,Employment,Employment,Employment,Employment,Employment,⋯,Employment,Employment,Employment,0,11,0,0,1,0,Steady EET


In [39]:
destinations_counts_labelled |>
    group_by(Label) |>
    tally()


Label,n
<chr>,<int>
DROP,2015
Risky trajectory,1291
Steady EET,14232


In [40]:
write.csv(destinations_counts_labelled, "data/additional_manual_classification.csv", row.names = FALSE)

## Extract person IDs

In [16]:
person_ids <- destinations_counts_labelled |>
    select(person_id, NCCIS_ACADYR)

In [17]:
head(person_ids)

person_id,NCCIS_ACADYR
<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2015/2016
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,2015/2016
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,2016/2017
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017


In [18]:
write.csv(person_ids, "data/additional_person_ids.csv", row.names = FALSE)